# Progetto Vendite
## Fasi del progetto
-  Gestione librerie
-  Importare dati e dataframe
-  Analisi esplorativa e descrittiva
-  Codifica dei dati
    - Uniformato il tipo di dati
    - Uniformato il contenuto dei dati
- Creazione nuove colonne
- Analisi riassuntiva
- Tabelle pivot
- Report Excel
- Prompt AI
- Documento Word con dati e analisi predittiva

## Gestione librerie

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from pandasai import SmartDataframe
from pandasai.llm.openai import OpenAI
from docx import Document
from docx.shared import Pt


## Importo dataframe

In [ ]:
df_vendite=pd.read_csv('vendite.csv', sep=';')

In [ ]:
df_vendite.head(5)

,data,cliente,città,prodotto,quantità,prezzo_unitario
0,01/01/2024,Cliente 013,Torino,Prodotto 004,6,"172,44"
1,01/01/2024,Cliente 036,Palermo,Prodotto 047,9,296
2,01/01/2024,Cliente 040,Bari,Prodotto 012,5,"41,37"
3,01/01/2024,Cliente 030,Potenza,Prodotto 031,9,"287,94"
4,01/01/2024,Cliente 032,Perugia,Prodotto 056,5,"42,1"


In [ ]:
df_vendite.tail(5)

,data,cliente,città,prodotto,quantità,prezzo_unitario
99995,30/09/2026,Cliente 047,Milano,Prodotto 044,7,"232,07"
99996,30/09/2026,Cliente 004,Trieste,Prodotto 093,8,"239,33"
99997,30/09/2026,Cliente 039,Cagliari,Prodotto 065,1,"162,66"
99998,30/09/2026,Cliente 011,Napoli,Prodotto 042,7,"152,4"
99999,30/09/2026,Cliente 028,Cagliari,Prodotto 032,8,"334,89"


## Analisi descrittiva

In [ ]:
df_vendite.shape

(100000, 6)

In [ ]:
df_vendite.columns

Index(['data', 'cliente', 'città', 'prodotto', 'quantità', 'prezzo_unitario'], dtype='object')

In [ ]:
df_vendite.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 6 columns):
 #   Column           Non-Null Count   Dtype 
---  ------           --------------   ----- 
 0   data             100000 non-null  object
 1   cliente          100000 non-null  object
 2   città            100000 non-null  object
 3   prodotto         100000 non-null  object
 4   quantità         100000 non-null  int64 
 5   prezzo_unitario  100000 non-null  object
dtypes: int64(1), object(5)
memory usage: 4.6+ MB


In [ ]:
df_vendite.describe(include='all')

,data,cliente,città,prodotto,quantità,prezzo_unitario
count,100000,100000,100000,100000,100000.000000,100000
unique,1004,50,20,100,NaN,42660
top,28/05/2024,Cliente 023,Bari,Prodotto 056,NaN,"96,27"
freq,133,2095,5098,1074,NaN,11
mean,NaN,NaN,NaN,NaN,5.497660,NaN
std,NaN,NaN,NaN,NaN,2.867308,NaN
min,NaN,NaN,NaN,NaN,1.000000,NaN
25%,NaN,NaN,NaN,NaN,3.000000,NaN
50%,NaN,NaN,NaN,NaN,5.000000,NaN
75%,NaN,NaN,NaN,NaN,8.000000,NaN


## Codifica dei dati

### Uniformare il contenuto delle colonne di tipo testo

In [ ]:
colonne_testo = ["cliente", "città", "prodotto"]
for colonna in colonne_testo:
    df_vendite[colonna] = df_vendite[colonna].astype(str).str.strip()
    df_vendite[colonna] = df_vendite[colonna].astype(str).str.lower()

In [ ]:
df_vendite.head(5)

,data,cliente,città,prodotto,quantità,prezzo_unitario
0,01/01/2024,cliente 013,torino,prodotto 004,6,"172,44"
1,01/01/2024,cliente 036,palermo,prodotto 047,9,296
2,01/01/2024,cliente 040,bari,prodotto 012,5,"41,37"
3,01/01/2024,cliente 030,potenza,prodotto 031,9,"287,94"
4,01/01/2024,cliente 032,perugia,prodotto 056,5,"42,1"


### Uniformare il contenuto delle colonne di tipo numero

In [ ]:
df_vendite['prezzo_unitario'] = df_vendite['prezzo_unitario'].str.replace(',', '.')
df_vendite['prezzo_unitario'] = pd.to_numeric(df_vendite['prezzo_unitario'], errors='coerce')

In [ ]:
df_vendite.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 6 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   data             100000 non-null  object 
 1   cliente          100000 non-null  object 
 2   città            100000 non-null  object 
 3   prodotto         100000 non-null  object 
 4   quantità         100000 non-null  int64  
 5   prezzo_unitario  100000 non-null  float64
dtypes: float64(1), int64(1), object(4)
memory usage: 4.6+ MB


### Uniformare il contenuto delle colonne di tipo data

In [ ]:
df_vendite['data'] = df_vendite['data'].str.replace('/', '-')
df_vendite['data']=pd.to_datetime(df_vendite['data'], format="%d-%m-%Y", errors='coerce')

In [ ]:
df_vendite.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 6 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   data             100000 non-null  datetime64[ns]
 1   cliente          100000 non-null  object        
 2   città            100000 non-null  object        
 3   prodotto         100000 non-null  object        
 4   quantità         100000 non-null  int64         
 5   prezzo_unitario  100000 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int64(1), object(3)
memory usage: 4.6+ MB


## Creazione nuove colonne

### Fatturato totale

In [ ]:
df_vendite['fatturato_totale'] = df_vendite['quantità'] * df_vendite['prezzo_unitario']

In [ ]:
df_vendite.head(5)

,data,cliente,città,prodotto,quantità,prezzo_unitario,fatturato_totale
0,2024-01-01,cliente 013,torino,prodotto 004,6,172.44,1034.64
1,2024-01-01,cliente 036,palermo,prodotto 047,9,296.00,2664.00
2,2024-01-01,cliente 040,bari,prodotto 012,5,41.37,206.85
3,2024-01-01,cliente 030,potenza,prodotto 031,9,287.94,2591.46
4,2024-01-01,cliente 032,perugia,prodotto 056,5,42.10,210.50


### Anno

In [ ]:
df_vendite['anno'] = df_vendite['data'].dt.year
df_vendite['anno1'] = df_vendite['data'].dt.to_period('Y')

In [ ]:
df_vendite.head(5)

,data,cliente,città,prodotto,quantità,prezzo_unitario,fatturato_totale,anno,anno1
0,2024-01-01,cliente 013,torino,prodotto 004,6,172.44,1034.64,2024,2024
1,2024-01-01,cliente 036,palermo,prodotto 047,9,296.00,2664.00,2024,2024
2,2024-01-01,cliente 040,bari,prodotto 012,5,41.37,206.85,2024,2024
3,2024-01-01,cliente 030,potenza,prodotto 031,9,287.94,2591.46,2024,2024
4,2024-01-01,cliente 032,perugia,prodotto 056,5,42.10,210.50,2024,2024


### Mese

In [ ]:
df_vendite['mese'] = df_vendite['data'].dt.month
df_vendite['mese1'] = df_vendite['data'].dt.to_period('M')


In [ ]:
df_vendite.head(5)

,data,cliente,città,prodotto,quantità,prezzo_unitario,fatturato_totale,anno,anno1,mese,mese1
0,2024-01-01,cliente 013,torino,prodotto 004,6,172.44,1034.64,2024,2024,1,2024-01
1,2024-01-01,cliente 036,palermo,prodotto 047,9,296.00,2664.00,2024,2024,1,2024-01
2,2024-01-01,cliente 040,bari,prodotto 012,5,41.37,206.85,2024,2024,1,2024-01
3,2024-01-01,cliente 030,potenza,prodotto 031,9,287.94,2591.46,2024,2024,1,2024-01
4,2024-01-01,cliente 032,perugia,prodotto 056,5,42.10,210.50,2024,2024,1,2024-01


## Riepiloghi ed Analisi riassuntive

### Fatturato per prodotto

In [ ]:
fatturato_per_prodotto = df_vendite.groupby('prodotto', as_index=False)['fatturato_totale'].sum().sort_values('fatturato_totale', ascending=False)

In [ ]:
fatturato_per_prodotto

,prodotto,fatturato_totale
21,prodotto 022,1525828.55
36,prodotto 037,1520903.02
31,prodotto 032,1504709.34
71,prodotto 072,1501412.81
80,prodotto 081,1490291.13
...,...,...
63,prodotto 064,1304970.37
93,prodotto 094,1304694.99
70,prodotto 071,1304435.22
30,prodotto 031,1291059.96


### Fatturato per anno

In [ ]:
fatturato_per_anno = df_vendite.groupby(['anno'], as_index=False)['fatturato_totale'].sum().sort_values('anno', ascending=True)

In [ ]:
fatturato_per_anno

,anno,fatturato_totale
0,2024,51173237.36
1,2025,50700146.67
2,2026,38342931.19


### Fatturato per mese-anno

In [ ]:
fatturato_per_mese_anno = df_vendite.groupby(['mese1'], as_index=False)['fatturato_totale'].sum().sort_values('mese1', ascending=True)

In [ ]:
fatturato_per_mese_anno

,mese1,fatturato_totale
0,2024-01,4348337.46
1,2024-02,3951723.54
2,2024-03,4408563.50
3,2024-04,4262801.73
4,2024-05,4134462.55
5,2024-06,4184319.34
6,2024-07,4317089.71
7,2024-08,4385815.31
8,2024-09,4293218.65
9,2024-10,4401089.12


### Fatturato per città

In [ ]:
fatturato_per_città = df_vendite.groupby(['città'], as_index=False)['fatturato_totale'].sum().sort_values('fatturato_totale', ascending=False)

In [ ]:
fatturato_per_città

,città,fatturato_totale
6,catanzaro,7244704.39
2,bari,7227428.80
18,trieste,7200660.90
14,potenza,7149763.47
12,palermo,7084383.90
15,roma,7065158.37
17,trento,7054331.53
5,campobasso,7052265.49
1,aosta,7037802.47
7,firenze,6981419.49


### Fatturato per cliente

In [ ]:
fatturato_per_cliente = df_vendite.groupby(['cliente'], as_index=False)['fatturato_totale'].sum().sort_values('fatturato_totale', ascending=False)

In [ ]:
fatturato_per_cliente

,cliente,fatturato_totale
43,cliente 044,2990823.97
25,cliente 026,2970886.21
44,cliente 045,2962222.87
32,cliente 033,2935811.05
19,cliente 020,2931936.20
26,cliente 027,2923179.87
14,cliente 015,2903161.75
22,cliente 023,2900248.83
10,cliente 011,2894053.88
39,cliente 040,2884659.72


## Tabelle pivot

### Tabella città prodotto

In [ ]:
pivot_citta_prodotto = pd.pivot_table(
   df_vendite,
   index='città',
   columns='prodotto',
   values='fatturato_totale',
   aggfunc='sum',
   fill_value=0
)

In [ ]:
pivot_citta_prodotto

prodotto,prodotto 001,prodotto 002,prodotto 003,prodotto 004,prodotto 005,prodotto 006,prodotto 007,prodotto 008,prodotto 009,prodotto 010,...,prodotto 091,prodotto 092,prodotto 093,prodotto 094,prodotto 095,prodotto 096,prodotto 097,prodotto 098,prodotto 099,prodotto 100
città,,,,,,,,,,,,,,,,,,,,,
ancona,55659.85,72460.99,83352.50,85359.50,69494.11,64220.24,57551.47,57064.12,63170.55,60113.16,...,88382.94,58891.55,83387.23,81086.19,75218.40,84293.97,64709.05,70523.38,48524.33,59387.90
aosta,76833.28,98434.80,62806.45,96341.01,52974.41,85962.31,76523.38,64929.97,77295.10,69553.82,...,65346.25,69453.50,83474.73,52375.19,83042.40,72558.36,61222.15,63981.98,71691.07,85102.51
bari,89310.58,68740.18,65457.24,88950.62,77246.98,61241.60,71656.36,68560.81,56109.58,76800.29,...,102843.84,66000.50,56274.98,78884.54,83782.73,55983.11,62082.49,66972.06,87532.03,61225.62
bologna,68966.17,97386.37,67264.73,52257.96,78068.74,109052.04,77700.37,73963.42,90240.93,54080.95,...,63901.43,44949.20,100866.47,39671.37,97116.12,69216.39,43839.54,63386.86,74470.42,89678.39
cagliari,56320.11,83123.53,78494.03,85094.52,72275.23,81668.13,67501.73,87275.30,68311.68,69961.95,...,67593.75,59927.91,86980.09,74513.49,56561.02,80319.68,65331.38,49836.07,64557.02,53663.46
campobasso,76280.29,62638.78,77437.02,61269.71,87744.84,60841.31,53450.57,85257.26,69772.49,79992.45,...,81400.14,56243.04,61911.97,58114.15,62019.96,61381.94,63870.08,72747.86,72577.32,58990.26
catanzaro,67602.37,52981.00,56547.49,71502.21,65154.97,65581.50,64800.14,93520.80,90531.09,89077.59,...,74005.02,66546.39,63191.42,62499.70,80600.43,66408.10,70688.23,106071.05,51057.97,67259.50
firenze,68396.75,95416.22,63052.82,56720.22,53363.31,52559.60,59841.64,68610.96,73351.16,76012.76,...,51378.36,60795.12,78354.61,57310.31,73312.13,68566.84,51327.23,75380.35,70222.64,73469.13
genova,69548.77,72131.94,64044.01,55853.90,67947.49,66501.29,78147.15,71753.03,76677.29,62905.97,...,82511.53,85723.27,62382.85,79859.25,76921.53,54191.99,76403.92,73166.56,59379.56,83137.24


### Per città anno

In [ ]:
pivot_citta_anno = pd.pivot_table(
   df_vendite,
   index='città',
   columns='anno',
   values='fatturato_totale',
   aggfunc='sum',
   fill_value=0
)
pivot_citta_anno

anno,2024,2025,2026
città,,,
ancona,2530857.45,2536100.13,1884815.65
aosta,2456006.31,2518942.82,2062853.34
bari,2629291.90,2711997.74,1886139.16
bologna,2525368.17,2441301.74,1880429.07
cagliari,2513112.67,2520103.94,1905598.82
campobasso,2564986.79,2540497.44,1946781.26
catanzaro,2639226.18,2627070.98,1978407.23
firenze,2452150.70,2654786.93,1874481.86
genova,2578912.75,2319513.27,2040875.42


### Per città anno-mese

In [ ]:
pivot_citta_anno_mese = pd.pivot_table(
   df_vendite,
   index='città',
   columns='mese1',
   values='fatturato_totale',
   aggfunc='sum',
   fill_value=0
)
pivot_citta_anno_mese

mese1,2024-01,2024-02,2024-03,2024-04,2024-05,2024-06,2024-07,2024-08,2024-09,2024-10,...,2025-12,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06,2026-07,2026-08,2026-09
città,,,,,,,,,,,,,,,,,,,,,
ancona,178534.12,192474.06,213095.49,232823.42,168668.81,193877.07,252096.23,220909.38,221017.71,223085.11,...,189590.81,213417.87,184017.25,223299.62,198375.29,214192.51,195283.07,223103.69,208162.46,224963.89
aosta,205543.98,176030.93,203145.00,219945.76,201320.52,261413.93,192094.95,193433.92,241332.52,188365.96,...,213922.51,210115.77,239516.88,236645.09,220879.18,221780.52,242585.61,194037.28,292397.48,204895.53
bari,200040.61,243646.19,212644.88,209287.42,215979.18,197617.24,196042.12,228266.81,281773.95,192070.15,...,248022.83,220124.87,207897.05,235571.39,202413.62,211001.50,225021.50,189547.74,228402.17,166159.32
bologna,232455.73,200101.49,221760.72,230549.55,188827.51,203164.02,187165.03,209395.33,212497.95,194784.75,...,217688.65,225372.16,195195.03,171839.85,241779.92,242173.05,203197.41,224680.75,205311.47,170879.43
cagliari,235636.60,188818.24,223125.38,167369.78,197021.40,232148.55,224419.57,203692.57,219680.90,216752.59,...,186895.56,208132.29,187246.31,194324.73,177862.59,246766.44,252788.69,190110.76,197293.66,251073.35
campobasso,207693.33,187954.18,216837.80,225628.57,211735.60,249634.35,192522.47,227629.09,161901.57,236547.97,...,221038.74,216999.63,189533.17,229666.69,219616.93,215150.28,227134.80,237584.39,234794.43,176300.94
catanzaro,221737.88,215291.96,213397.16,229016.25,197619.09,219619.18,199427.15,255367.27,237963.66,238814.44,...,182611.20,206679.42,185510.33,236284.30,204091.68,195393.97,242030.11,231248.88,231793.72,245374.82
firenze,197543.01,211129.86,201524.56,214292.80,210785.35,187975.69,195091.19,200288.72,237941.13,233671.81,...,210888.00,187778.63,190240.43,227586.64,222753.68,209194.65,183459.02,203492.31,261323.58,188652.92
genova,233963.03,209362.99,196286.38,218077.26,210786.68,179538.73,213850.47,239333.51,183442.23,211379.33,...,207669.49,203579.14,184638.01,270890.67,215486.72,250446.37,231645.43,222325.64,227216.16,234647.28


### Per prodotto anno

In [ ]:
pivot_anno_prodotto = pd.pivot_table(
   df_vendite,
   index='anno',
   columns='prodotto',
   values='fatturato_totale',
   aggfunc='sum',
   fill_value=0
)
pivot_anno_prodotto

prodotto,prodotto 001,prodotto 002,prodotto 003,prodotto 004,prodotto 005,prodotto 006,prodotto 007,prodotto 008,prodotto 009,prodotto 010,...,prodotto 091,prodotto 092,prodotto 093,prodotto 094,prodotto 095,prodotto 096,prodotto 097,prodotto 098,prodotto 099,prodotto 100
anno,,,,,,,,,,,,,,,,,,,,,
2024,528509.62,481608.40,568234.15,565704.67,515759.08,502320.84,481317.74,509742.55,536972.29,495386.18,...,507680.17,536881.07,496739.27,465470.67,475610.16,489016.98,472526.73,493679.69,491389.25,535332.67
2025,516126.25,498252.57,508346.80,474119.10,520903.96,517327.08,502118.07,567046.40,486372.79,535739.05,...,517865.45,420974.49,542987.20,446988.35,552588.68,540162.01,525617.30,518919.25,542644.76,475815.44
2026,334478.08,455281.31,345731.33,392317.38,354559.28,397827.16,387855.85,363923.61,366693.02,363102.51,...,456529.86,370575.59,397714.77,392235.97,408342.97,352818.04,346321.77,377635.51,357096.42,409399.45


### Per prodotto anno-mese

In [ ]:
pivot_mese1_prodotto = pd.pivot_table(
   df_vendite,
   index='mese1',
   columns='prodotto',
   values='fatturato_totale',
   aggfunc='sum',
   fill_value=0
)
pivot_mese1_prodotto

prodotto,prodotto 001,prodotto 002,prodotto 003,prodotto 004,prodotto 005,prodotto 006,prodotto 007,prodotto 008,prodotto 009,prodotto 010,...,prodotto 091,prodotto 092,prodotto 093,prodotto 094,prodotto 095,prodotto 096,prodotto 097,prodotto 098,prodotto 099,prodotto 100
mese1,,,,,,,,,,,,,,,,,,,,,
2024-01,20768.61,43762.45,41275.24,62205.88,48970.58,58336.60,41832.00,44233.09,57542.21,45781.51,...,41676.05,52574.57,30756.66,31011.73,31884.77,37206.06,45320.85,44396.47,27636.61,55502.77
2024-02,53507.84,26083.50,45168.49,47359.08,44621.03,49346.14,30112.79,42516.27,48503.80,23140.33,...,34081.76,35542.26,35908.28,33349.33,21224.38,51414.96,33185.62,47654.25,49296.72,42701.53
2024-03,34563.12,39479.42,43499.06,31287.53,37762.05,35149.61,40415.73,24647.03,50998.82,41652.48,...,46402.31,50143.64,46567.64,48914.56,49932.29,36874.39,54587.63,41386.74,38796.22,52033.90
2024-04,43506.76,43425.27,41480.44,36896.90,38244.04,40752.77,43890.89,34437.83,52285.45,54949.70,...,40434.84,24262.04,46602.52,29874.88,50286.54,49062.16,42735.09,20203.32,39660.37,45497.11
2024-05,47627.34,49425.05,53223.44,47255.17,44029.52,44547.00,28818.51,39395.16,52029.67,40586.45,...,51350.41,44149.12,32449.34,37100.89,42697.48,40373.67,15202.26,29763.63,44445.70,28760.55
2024-06,53038.38,29686.75,41930.88,38996.25,52836.08,32210.05,37564.51,49236.62,27254.88,53534.53,...,47960.27,31679.16,47618.31,30838.73,49323.00,32145.97,27925.33,44993.38,28726.30,62383.52
2024-07,41107.75,38965.04,55164.67,69426.80,52234.50,38001.87,46497.45,60468.44,58358.03,45369.18,...,48763.17,55555.38,41845.25,41128.63,38614.19,43243.76,42722.55,38955.62,45481.32,52735.09
2024-08,42082.96,64503.94,56360.66,53440.81,31812.05,37185.41,38200.54,59314.97,42883.30,43945.76,...,35752.54,40780.38,38392.95,24526.14,39966.17,48723.66,40216.97,34630.90,55515.30,48281.45
2024-09,44050.35,26445.74,56247.59,55167.08,46831.48,52598.03,29597.94,29537.21,49132.21,32676.21,...,41619.67,70644.82,49658.67,51168.93,42748.41,47696.41,60939.83,41014.71,42885.99,32018.35


## Report Excel multi-foglio

In [ ]:
nome_report = f"report_vendite_{datetime.now().strftime('%Y%m%d')}.xlsx"

with pd.ExcelWriter(nome_report, engine='openpyxl') as writer:
    df_vendite.to_excel(writer, sheet_name='Dati puliti', index=False)
    fatturato_per_prodotto.to_excel(writer, sheet_name='Per prodotto', index=False)
    fatturato_per_città.to_excel(writer, sheet_name='Per città', index=False)
    fatturato_per_mese_anno.to_excel(writer, sheet_name='Per mese', index=False)
    fatturato_per_anno.to_excel(writer, sheet_name='Per anno', index=False)
    pivot_citta_prodotto.to_excel(writer, sheet_name='Pivot città-prodotto')
    pivot_mese1_prodotto.to_excel(writer, sheet_name='Pivot mese1-prodotto')
    pivot_anno_prodotto.to_excel(writer, sheet_name='Pivot anno-prodotto')
    pivot_citta_anno.to_excel(writer, sheet_name='Pivot città-anno')
    pivot_citta_anno_mese.to_excel(writer, sheet_name='Pivot città-anno-mese')

nome_report

'report_vendite_20261023.xlsx'

## Prompt per l’AI
Usa solo dati aggregati, non i dati grezzi con clienti identificabili:

In [ ]:
totale = df_vendite['fatturato_totale'].sum()

prodotto_top = fatturato_per_prodotto.loc[fatturato_per_prodotto['fatturato_totale'].idxmax(), 'prodotto']
citta_top = fatturato_per_città.loc[fatturato_per_città['fatturato_totale'].idxmax(), 'città']

prompt_ai = f"""
Comportati come un analista dati aziendale.

Analizza i seguenti dati di vendita aggregati.

Fatturato totale: {totale:.2f} euro
Prodotto più redditizio: {prodotto_top}
Città con maggior fatturato: {citta_top}

Dati per prodotto:
{fatturato_per_prodotto.to_string(index=False)}

Dati per città:
{fatturato_per_città.to_string(index=False)}

Dati per mese:
{fatturato_per_mese_anno.to_string(index=False)}

Genera:
1. Tre insight principali.
2. Due rischi o anomalie.
3. Tre azioni commerciali concrete.
4. Una previsione qualitativa per il prossimo periodo.

Usa un linguaggio semplice, professionale e operativo.
"""

print(prompt_ai)

with open('prompt_ai.txt', 'w', encoding='utf-8') as f:
    f.write(prompt_ai)


Comportati come un analista dati aziendale.

Analizza i seguenti dati di vendita aggregati.

Fatturato totale: 140216315.22 euro
Prodotto più redditizio: prodotto 022
Città con maggior fatturato: catanzaro

Dati per prodotto:
    prodotto  fatturato_totale
prodotto 022        1525828.55
prodotto 037        1520903.02
prodotto 032        1504709.34
prodotto 072        1501412.81
prodotto 081        1490291.13
prodotto 055        1487334.02
prodotto 079        1484791.70
prodotto 015        1483131.15
prodotto 091        1482075.48
prodotto 040        1474125.94
prodotto 054        1463732.59
prodotto 059        1462502.44
prodotto 066        1460757.02
prodotto 065        1460452.63
prodotto 045        1456772.42
prodotto 088        1455173.30
prodotto 087        1453487.80
prodotto 051        1451763.66
prodotto 029        1444649.75
prodotto 063        1442613.39
prodotto 056        1442263.13
prodotto 008        1440712.56
prodotto 016        1440138.13
prodotto 084        1439628.0

Il portafoglio è molto bilanciato: nessun prodotto o città concentra il fatturato, quindi il rischio principale non è la dipendenza da pochi prodotti, ma la difficoltà di individuare leve di crescita differenziate. Il fatturato totale è di 140,2 milioni di euro, con un andamento mensile stabile ma una leggera flessione annua nel 2025.

## Tre insight principali

- **Portafoglio prodotti estremamente omogeneo.** Il prodotto 022 guida con 1,53 milioni di euro, ma il divario rispetto all’ultimo prodotto (077) è solo del 18,3%; i primi 10 prodotti generano appena il 10,7% del totale. Questo indica una domanda distribuita, con pochi “campioni” netti da scalare rapidamente.

- **Catanzaro guida, ma le città sono quasi alla pari.** Catanzaro fattura 7,24 milioni, seguita da vicino da Bari e Trieste; Milano è ultima con 6,79 milioni. La forbice tra prima e ultima città è solo del 6,7%, quindi non esistono mercati fortemente sottoperformanti o dominanti.

- **Andamento mensile stabile, con stagionalità marcata a febbraio.** Febbraio è sistematicamente il mese più debole: in media 3,90 milioni contro 4,28 milioni degli altri mesi, cioè circa il 9% in meno. Agosto e ottobre sono invece tra i mesi più solidi, con medie rispettivamente di 4,44 e 4,40 milioni di euro.

## Due rischi o anomalie

- **Flessione del 2025.** Il fatturato 2025 è sceso dello 0,9% rispetto al 2024, passando da 51,17 a 50,70 milioni di euro. Non è un crollo, ma segnala una crescita ferma che merita indagine su prezzi, mix, churn clienti o pressione competitiva.

- **Febbraio ricorrentemente sotto target.** Il calo di febbraio si ripete per tre anni consecutivi, con il minimo assoluto nel 2026 (3,81 milioni). Potrebbe dipendere da stagione, calendario vendite, rinnovi o minore attività commerciale, ma va verificato perché crea un buco prevedibile nel flusso di fatturato.

## Tre azioni commerciali concrete

- **Lanciare una campagna “Febbraio” strutturata.** Attivare promozioni, bundle, incentivi per la forza vendita e campagne di riattivazione clienti già da gennaio, con obiettivo di recuperare almeno metà del gap medio di circa 380 mila euro mensili.

- **Replicare le pratiche di Catanzaro nelle città medie.** Analizzare clienti, canali e prodotti che sostengono Catanzaro, Bari e Trieste, quindi trasferire playbook commerciali mirati a Milano, Bologna e Napoli, che oggi sono le tre città più basse.

- **Creare un programma di crescita per i prodotti 077, 031, 071, 094 e 064.** Sono gli ultimi cinque per fatturato; invece di tagliarli, testare pricing, posizionamento, formazione venditori e offerte mirate per capire se il problema è domanda, visibilità o esecuzione commerciale.

## Previsione qualitativa

Per il prossimo periodo ci si attende una **continuità dei volumi, senza una crescita forte automatica**: i primi nove mesi del 2026 sono leggermente sopra il medesimo periodo del 2025 (+0,7%), ma l’andamento resta piatto.  Se non si interviene su febbraio e sulle città a minore performance, il trimestre in chiusura dovrebbe restare stabile, con ottobre potenzialmente tra i mesi migliori; una campagna mirata su febbraio 2027 può invece trasformare la stagionalità negativa in un’opportunità di crescita.

## Documento Word

In [ ]:
from docx import Document
doc = Document()
doc.add_heading('Report vendite automatico', level=1)
doc.add_paragraph(f"Data di generazione: {datetime.now().strftime('%d/%m/%Y %H:%M')}")

doc.add_heading('Indicatori principali', level=2)
doc.add_paragraph(f"Fatturato totale: {totale:,.2f} euro")
doc.add_paragraph(f"Prodotto più redditizio: {prodotto_top}")
doc.add_paragraph(f"Città con maggior fatturato: {citta_top}")

doc.add_heading('Fatturato per prodotto', level=2)

tabella = doc.add_table(rows=1, cols=2)
tabella.style = 'Table Grid'

intestazione = tabella.rows[0].cells
intestazione[0].text = 'Prodotto'
intestazione[1].text = 'Fatturato'

for _, riga in fatturato_per_prodotto.iterrows():
    celle = tabella.add_row().cells
    celle[0].text = str(riga['prodotto'])
    celle[1].text = f"{riga['fatturato']:,.2f}"

doc.add_heading('Analisi e suggerimenti', level=2)
doc.add_paragraph(
    "L’analisi evidenzia i prodotti e le aree geografiche con maggiore contributo al fatturato. "
    "Si consiglia di monitorare mensilmente l’andamento e di concentrare le attività promozionali "
    "sui segmenti più redditizi."
)

doc.save('report_vendite.docx')

ModuleNotFoundError: No module named 'docx'